In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import getpass
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile

from google.colab import drive

drive.mount('/content/drive')

WORKSPACE = Path('/content')
PROJECT_DIR = WORKSPACE / 'point_cloud_playground_play'
REPO_URL = 'https://github.com/kslannmnd/point_cloud_playground_play.git'
GITHUB_TOKEN = ''
GITHUB_TOKEN_ENV = 'GITHUB_TOKEN'
DRIVE_ROOT = Path('/content/drive/MyDrive')
S3DIS_ZIP_PATH = DRIVE_ROOT / 'point_cloud_playground_play' / 'S3DIS_preprocessed.zip'
S3DIS_PREPROCESSED_DIR = WORKSPACE / 's3dis_preprocessed'
R3D_FILE_NAME = '2026-05-06--12-50-38.r3d'
R3D_DRIVE_PATH = DRIVE_ROOT / 'point_cloud_playground_play' / R3D_FILE_NAME
TEST_AREA = 'Area_5'
N_TEST_ROOMS = 3
N_TRAIN_ROOMS = 6
PRETRAINED_METRIC_VARIANT = 'n_rooms'
TRAIN_VARIANT = 'n_rooms'
TRAINED_METRIC_VARIANT = 'n_rooms'
TRAIN_EXPERIMENT_NAME = 'colab_softgroup_s3dis'

def run(cmd, cwd=None, env=None, check=True, display_cmd=None):
    text = display_cmd or (' '.join(map(str, cmd)) if isinstance(cmd, (list, tuple)) else str(cmd))
    print('\n[RUN]', text, '\n')
    proc = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        shell=isinstance(cmd, str),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if proc.stdout:
        print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f'Command failed with code {proc.returncode}:\n{text}')
    return proc

def get_github_token():
    token = GITHUB_TOKEN.strip() or os.environ.get(GITHUB_TOKEN_ENV)
    if not token:
        token = getpass.getpass('GitHub token for private repo: ')
    token = token.strip()
    if not token:
        raise RuntimeError('GitHub token is empty.')
    return token

def authenticated_repo_url():
    return REPO_URL.replace('https://', f'https://{get_github_token()}@', 1)

assert S3DIS_ZIP_PATH.exists(), S3DIS_ZIP_PATH

## Clone and Install

In [ ]:
clone_url = authenticated_repo_url()
if PROJECT_DIR.exists():
    print('Project already exists:', PROJECT_DIR)
    run(['git', 'remote', 'set-url', 'origin', clone_url], cwd=PROJECT_DIR, display_cmd='git remote set-url origin https://***@github.com/kslannmnd/point_cloud_playground_play.git')
    run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
else:
    run(['git', 'clone', clone_url, str(PROJECT_DIR)], display_cmd=f'git clone https://***@github.com/kslannmnd/point_cloud_playground_play.git {PROJECT_DIR}')

run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=PROJECT_DIR, check=False, display_cmd='git remote set-url origin https://github.com/kslannmnd/point_cloud_playground_play.git')

run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip', 'wheel', 'setuptools'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=PROJECT_DIR)

## Extract Preprocessed S3DIS

In [ ]:
def find_preprocessed_root(path):
    required = ['preprocess', 'preprocess_sample', 'val_gt']
    if all((path / name).is_dir() for name in required):
        return path
    candidates = [item for item in path.rglob('*') if item.is_dir()]
    for item in candidates:
        if all((item / name).is_dir() for name in required):
            return item
    return None

s3dis_root = find_preprocessed_root(S3DIS_PREPROCESSED_DIR)
if s3dis_root is None:
    S3DIS_PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(S3DIS_ZIP_PATH, 'r') as archive:
        archive.extractall(S3DIS_PREPROCESSED_DIR)
    s3dis_root = find_preprocessed_root(S3DIS_PREPROCESSED_DIR)

assert s3dis_root is not None, S3DIS_PREPROCESSED_DIR
S3DIS_PREPROCESSED_DIR = s3dis_root
print('S3DIS preprocessed root:', S3DIS_PREPROCESSED_DIR)

## Room Sets

In [ ]:
def natural_key(value):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r'(\d+)', str(value))]

def discover_rooms(preprocess_dir):
    rooms = []
    for path in preprocess_dir.glob('*_inst_nostuff.pth'):
        match = re.match(r'^(Area_\d+)_(.+)_inst_nostuff\.pth$', path.name)
        if match:
            area, room = match.group(1), match.group(2)
            rooms.append({'area': area, 'room': room, 'scene': f'{area}_{room}'})
    return sorted(rooms, key=lambda item: (natural_key(item['area']), natural_key(item['room'])))

def room_ref(room):
    return f"{room['area']}/{room['room']}"

def list_override(name, rooms):
    return f"{name}=[{','.join(room_ref(room) for room in rooms)}]"

def metric_room_args(variant):
    if variant == 'all_test_rooms':
        return ['metrics.rooms=all_test_rooms']
    if variant == 'n_rooms':
        return [list_override('metrics.rooms', all_test_rooms[:N_TEST_ROOMS])]
    raise ValueError(variant)

def train_room_args(variant):
    if variant == 'all_trainable_rooms':
        return ['training.rooms=all_trainable_rooms']
    if variant == 'n_rooms':
        return [list_override('training.rooms', all_trainable_rooms[:N_TRAIN_ROOMS])]
    raise ValueError(variant)

all_rooms = discover_rooms(S3DIS_PREPROCESSED_DIR / 'preprocess')
all_test_rooms = [room for room in all_rooms if room['area'] == TEST_AREA]
all_trainable_rooms = [room for room in all_rooms if room['area'] != TEST_AREA]
assert all_test_rooms, TEST_AREA
assert all_trainable_rooms, TEST_AREA
print('all_test_rooms:', len(all_test_rooms))
print('all_trainable_rooms:', len(all_trainable_rooms))
print('metric n rooms:', [room['scene'] for room in all_test_rooms[:N_TEST_ROOMS]])
print('train n rooms:', [room['scene'] for room in all_trainable_rooms[:N_TRAIN_ROOMS]])

## Colab CUDA Setup

In [ ]:
import subprocess
import torch

SPCONV_TAG_FALLBACKS = ['cu128', 'cu126', 'cu124', 'cu121', 'cu120', 'cu118', 'cu117', 'cu116']
SPCONV_CLEANUP_PACKAGES = ['cumm', 'spconv'] + [f'cumm-{tag}' for tag in SPCONV_TAG_FALLBACKS] + [f'spconv-{tag}' for tag in SPCONV_TAG_FALLBACKS]

def cuda_package_tag(cuda_version):
    if not cuda_version:
        raise RuntimeError('Colab runtime has no CUDA-enabled Torch. Switch Runtime type to GPU.')
    parts = cuda_version.split('.')
    return f"cu{parts[0]}{parts[1]}"

def cuda_version_from_tag(cuda_tag):
    digits = cuda_tag.removeprefix('cu')
    return f'{digits[:-1]}.{digits[-1]}'

def pip_distribution_exists(package):
    proc = subprocess.run(
        [sys.executable, '-m', 'pip', 'index', 'versions', package],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    output = proc.stdout or ''
    return proc.returncode == 0 and 'No matching distribution found' not in output

def select_spconv_stack(cuda_tag):
    tags = [cuda_tag] + [tag for tag in SPCONV_TAG_FALLBACKS if tag != cuda_tag]
    checked = []
    for tag in tags:
        cumm_package = f'cumm-{tag}'
        spconv_package = f'spconv-{tag}'
        cumm_ok = pip_distribution_exists(cumm_package)
        spconv_ok = pip_distribution_exists(spconv_package)
        checked.append((tag, cumm_ok, spconv_ok))
        if cumm_ok and spconv_ok:
            return cumm_package, spconv_package, tag, checked
    raise RuntimeError(f'No compatible cumm/spconv wheel pair found. Checked: {checked}')

COLAB_CUDA_VERSION = torch.version.cuda
COLAB_CUDA_TAG = cuda_package_tag(COLAB_CUDA_VERSION)
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    COLAB_CUDA_ARCH = f'{major}.{minor}'
else:
    raise RuntimeError('Colab runtime has no CUDA GPU. Switch Runtime type to GPU.')

COLAB_CUMM_PACKAGE, COLAB_SPCONV_PACKAGE, COLAB_SPCONV_TAG, COLAB_SPCONV_CHECKED = select_spconv_stack(COLAB_CUDA_TAG)
COLAB_CUMM_CUDA_VERSION = cuda_version_from_tag(COLAB_SPCONV_TAG)
print('Torch:', torch.__version__)
print('CUDA:', COLAB_CUDA_VERSION)
print('GPU arch:', COLAB_CUDA_ARCH)
print('checked spconv tags:', COLAB_SPCONV_CHECKED)
print('spconv package:', COLAB_SPCONV_PACKAGE)
print('cumm package:', COLAB_CUMM_PACKAGE)

run('apt-get update && apt-get install -y libsparsehash-dev ninja-build', cwd='/')
run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-U', 'ninja', 'packaging', 'numpy<2', 'gdown'])
run([sys.executable, '-m', 'pip', 'uninstall', '-y', *SPCONV_CLEANUP_PACKAGES], check=False)

## Environment and Preprocessed S3DIS Link

In [ ]:
COLAB_RUNTIME_OVERRIDES = [
    'env=colab',
    'env.is_kaggle=false',
    'env.use_apt=false',
    'env.torch.target_torch=',
    'env.torch.target_torchvision=',
    'env.torch.target_torchaudio=',
    'env.torch.index_url=',
    f'env.spconv.cumm={COLAB_CUMM_PACKAGE}',
    f'env.spconv.spconv={COLAB_SPCONV_PACKAGE}',
    f'softgroup.runtime_env.CUMM_CUDA_VERSION={COLAB_CUMM_CUDA_VERSION}',
    f'softgroup.runtime_env.CUMM_CUDA_ARCH_LIST={COLAB_CUDA_ARCH}',
    f'softgroup.runtime_env.TORCH_CUDA_ARCH_LIST={COLAB_CUDA_ARCH}',
    '++softgroup.runtime_env.MAX_JOBS=2',
]

COMMON = [
    *COLAB_RUNTIME_OVERRIDES,
    'dataset=s3dis_preprocessed',
    'inference=s3dis',
    f'env.kaggle.input_s3dis_preprocessed_dir={S3DIS_PREPROCESSED_DIR}',
    f'dataset.preprocessed.source_dir={S3DIS_PREPROCESSED_DIR}',
    f'dataset.preprocessed.archive_path={S3DIS_ZIP_PATH}',
    f'dataset.preprocessed.extract_dir={S3DIS_PREPROCESSED_DIR}',
    f'dataset.fold.preferred_test_area={TEST_AREA}',
]

R3D_COMMON = [
    *COLAB_RUNTIME_OVERRIDES,
    'dataset=r3d',
    'inference=r3d',
]

run([sys.executable, 'scripts/prepare_kaggle_env.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/install_dependencies_and_build_softgroup.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/prepare_s3dis.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/download_softgroup_checkpoints.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/patch_softgroup_for_kaggle.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/generate_softgroup_configs.py', *COMMON], cwd=PROJECT_DIR)

## 1. Pretrained Metrics on Test Rooms

In [ ]:
PRETRAINED_CHECKPOINT = 'outputs/checkpoints/softgroup_s3dis_pretrained.pth'
pretrained_metric_cmd = [
    sys.executable, 'scripts/compute_s3dis_metrics.py', *COMMON,
    f'metrics.checkpoint={PRETRAINED_CHECKPOINT}',
    'metrics.force_run=true',
    'metrics.allow_precomputed_results=false',
    *metric_room_args(PRETRAINED_METRIC_VARIANT),
]
run(pretrained_metric_cmd, cwd=PROJECT_DIR)

## Pretrained S3DIS and R3D Inference with BBox HTML

In [ ]:
S3DIS_ROOM = all_test_rooms[0]

def newest_run_dir(output_root):
    runs = sorted([path for path in output_root.iterdir() if path.is_dir()])
    assert runs, output_root
    return runs[-1]

def run_s3dis_bbox_inference(checkpoint, label):
    output_root = PROJECT_DIR / 'outputs' / 'inference' / label
    run([
        sys.executable, 'scripts/infer.py', *COMMON,
        f'inference.checkpoint={checkpoint}',
        f'inference.output_dir=outputs/inference/{label}',
        'inference.target.kind=room',
        f"inference.target.area={S3DIS_ROOM['area']}",
        f"inference.target.room={S3DIS_ROOM['room']}",
        'visualization.force_run=true',
        'visualization.allow_precomputed_results=false',
    ], cwd=PROJECT_DIR)
    run_dir = newest_run_dir(output_root)
    html_files = sorted((run_dir / 'html').glob('*.html'))
    assert html_files, run_dir
    html_path = html_files[-1]
    print(f'{label} S3DIS room:', S3DIS_ROOM['scene'])
    print(f'{label} S3DIS HTML:', html_path)
    display(HTML(html_path.read_text(encoding='utf-8')))
    return html_path

def ensure_r3d_file():
    r3d_file = PROJECT_DIR / 'data' / 'r3d' / R3D_FILE_NAME
    if not r3d_file.exists() and R3D_DRIVE_PATH.exists():
        r3d_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(R3D_DRIVE_PATH, r3d_file)
    assert r3d_file.exists(), r3d_file
    return r3d_file

def run_r3d_bbox_inference(checkpoint, label):
    r3d_file = ensure_r3d_file()
    output_root = PROJECT_DIR / 'outputs' / 'r3d_inference' / label
    run([
        sys.executable, 'run_r3d_inference.py', str(r3d_file), *R3D_COMMON,
        f'inference.checkpoint={checkpoint}',
        f'inference.output_dir=outputs/r3d_inference/{label}',
    ], cwd=PROJECT_DIR)
    run_dir = newest_run_dir(output_root)
    html_path = run_dir / 'point_cloud.html'
    assert html_path.exists(), html_path
    provenance_path = run_dir / 'provenance.json'
    provenance = json.loads(provenance_path.read_text(encoding='utf-8'))
    predicted_instances = int(provenance.get('predicted_instance_count') or 0)
    assigned_points = int(provenance.get('assigned_instance_points') or 0)
    assert predicted_instances > 0, provenance_path
    assert assigned_points > 0, provenance_path
    print(f'{label} R3D predicted instances:', predicted_instances)
    print(f'{label} R3D assigned instance points:', assigned_points)
    print(f'{label} R3D instance summary:', provenance.get('instance_summary_csv'))
    print(f'{label} R3D point assignments:', provenance.get('point_instance_csv'))
    print(f'{label} R3D HTML:', html_path)
    display(HTML(html_path.read_text(encoding='utf-8')))
    return html_path

pretrained_s3dis_html_path = run_s3dis_bbox_inference(PRETRAINED_CHECKPOINT, 'pretrained')
pretrained_r3d_html_path = run_r3d_bbox_inference(PRETRAINED_CHECKPOINT, 'pretrained')

## 2. Train SoftGroup on Trainable Rooms

In [ ]:
train_cmd = [
    sys.executable, 'scripts/train.py', *COMMON,
    'training=exp_1',
    f'training.experiment_name={TRAIN_EXPERIMENT_NAME}',
    'training.epochs.backbone=2',
    'training.epochs.full_softgroup=2',
    'training.lr.backbone=0.001',
    'training.lr.full_softgroup=0.001',
    'training.train_batch_size=1',
    'training.test_batch_size=1',
    'training.num_workers=2',
    *train_room_args(TRAIN_VARIANT),
]
run(train_cmd, cwd=PROJECT_DIR)
TRAINED_CHECKPOINT = f'outputs/checkpoints/{TRAIN_EXPERIMENT_NAME}_full_softgroup_latest.pth'
assert (PROJECT_DIR / TRAINED_CHECKPOINT).exists(), PROJECT_DIR / TRAINED_CHECKPOINT
print('Trained checkpoint:', PROJECT_DIR / TRAINED_CHECKPOINT)

## 3. Trained Metrics on Test Rooms

In [ ]:
trained_metric_cmd = [
    sys.executable, 'scripts/compute_s3dis_metrics.py', *COMMON,
    f'metrics.checkpoint={TRAINED_CHECKPOINT}',
    'metrics.force_run=true',
    'metrics.allow_precomputed_results=false',
    *metric_room_args(TRAINED_METRIC_VARIANT),
]
run(trained_metric_cmd, cwd=PROJECT_DIR)

## 4. Trained S3DIS Test Room Inference with BBox HTML

In [ ]:
trained_s3dis_html_path = run_s3dis_bbox_inference(TRAINED_CHECKPOINT, 'trained')

## 5. Trained R3D Frame Inference with Red-Point Boxes

In [ ]:
trained_r3d_html_path = run_r3d_bbox_inference(TRAINED_CHECKPOINT, 'trained')

## Saved Artifacts

In [ ]:
metrics_root = PROJECT_DIR / 'outputs' / 'metrics' / 's3dis'
metric_runs = sorted([path for path in metrics_root.iterdir() if path.is_dir()]) if metrics_root.exists() else []
print('Metric runs:', len(metric_runs))
for path in metric_runs[-3:]:
    summary_path = path / 'summary.json'
    print(path)
    if summary_path.exists():
        print(json.loads(summary_path.read_text(encoding='utf-8')))
print('Pretrained checkpoint:', PROJECT_DIR / PRETRAINED_CHECKPOINT)
print('Trained checkpoint:', PROJECT_DIR / TRAINED_CHECKPOINT)
print('Pretrained S3DIS HTML:', pretrained_s3dis_html_path)
print('Pretrained R3D HTML:', pretrained_r3d_html_path)
print('Trained S3DIS HTML:', trained_s3dis_html_path)
print('Trained R3D HTML:', trained_r3d_html_path)